In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from IPython.display import Audio
import IPython.display as ipd
from scipy.io import wavfile
import tempfile
import os
import librosa
import pandas as pd
import seaborn as sns
import h5py
import mne
from scipy.stats import zscore
from mne_bids import BIDSPath, read_raw_bids
from matplotlib_venn import venn2,venn2_circles
from tqdm import tqdm

In [10]:
cm = 1/2.54
plt.rcParams['svg.fonttype'] = 'none'

fontdict = dict(fontsize=7)
fontsize = 7

red = '#A9373B'
blue = '#2369BD'
orange = '#CC8963'
green = '#009944'

stg_color = '#20B2AA'
smc_color = '#6A5ACD'
insula_color = '#D4AF37'

reds = sns.light_palette(red, as_cmap=True)
blues = sns.light_palette(blue, as_cmap=True)
oranges = sns.light_palette(orange, as_cmap=True)
greens = sns.light_palette(green, as_cmap=True)

recon_dir = '/cwork/ns458/ECoG_Recon/'
mne.viz.set_3d_backend('notebook')                    

'notebook'

In [11]:
import pandas as pd
import numpy as np
from mne_bids import BIDSPath
from tqdm import tqdm

task = [
        'LexicalNoDelay',
        'LexicalDelay',
        'PictureNaming',
        'SentenceRep',
        ]


ref = 'bipolar'
hga_paths = []
for t in task:
    hga_paths.extend(
        BIDSPath(
            root=f'../results/{t}({ref})',
            datatype='HGA',
            suffix='time',  # 注意这里改成 'time'
            check=False,
        ).match()
    )

# 2. 筛选 perception 和 passive 的数据
HGAs = []
for path in tqdm(hga_paths):
    df = pd.read_csv(path)
    # 只保留 perception 或 passive
    HGAs.append(df)

HGAs = pd.concat(HGAs)

# rename both phase : Resp and Response to Resp
HGAs.loc[HGAs.phase == 'Resp', 'phase'] = 'Response'
HGAs.loc[HGAs.phase == 'Audio', 'phase'] = 'Stimulus'

# phase all convert to lower
HGAs['phase'] = HGAs['phase'].str.lower()

# rename INS to 'Insula'
HGAs.loc[HGAs.roi == 'INS', 'roi'] = 'Insula'
# combine HG to STG
# HGAs.loc[HGAs.roi == 'HG', 'roi'] = 'STG'
HGAs.loc[HGAs.roi == 'PrG', 'roi'] = 'SMC'
HGAs.loc[HGAs.roi == 'PoG', 'roi'] = 'SMC'
HGAs.loc[HGAs.roi == 'Subcentral', 'roi'] = 'SMC'


HGAs.loc[HGAs.label.isin([
    'ctx_lh_G_and_S_cingul-Mid-Post',
    'ctx_rh_G_and_S_cingul-Mid-Post',
    'ctx_lh_G_cingul-Post-dorsal'
    ]), 'roi'] = 'PCC'

# 这种方案下，我们将 ROI 直接命名为 dACC，强调其在显著性网络中的角色
dacc_labels = [
    'ctx_lh_G_and_S_cingul-Mid-Ant',
    'ctx_rh_G_and_S_cingul-Mid-Ant'
]

dacc_labels = [
    'ctx_lh_G_and_S_cingul-Mid-Ant',
    'ctx_rh_G_and_S_cingul-Mid-Ant'
]
HGAs.loc[HGAs.label.isin(dacc_labels), 'roi'] = 'dACC' # 这里的 dACC 就是最严格的 SN 枢纽
# rename CG to ACC
HGAs.loc[HGAs.roi=='CG', 'roi'] = 'ACC'
HGAs.head()

100%|██████████| 1005/1005 [00:11<00:00, 89.00it/s]


,time,channel,value,mask,roi,hemi,subject,description,task,phase,modality,label,x,y,z
0,-1.0,D0024_LTG14-15,0.007910,False,STG,L,D0024,Decision,LexicalNoDelay,stimulus,sound,ctx_lh_G_temp_sup-Lateral,-66.294399,-28.103954,7.090004
1,-1.0,D0024_LTG8-9,-0.001315,False,Intersection,L,D0024,Decision,LexicalNoDelay,stimulus,sound,Intersection,-55.427594,-7.961854,2.275063
2,-1.0,D0024_LTG5-6,0.026363,False,SMC,L,D0024,Decision,LexicalNoDelay,stimulus,sound,ctx_lh_G_and_S_subcentral,-65.187402,-14.882005,13.114168
3,-1.0,D0024_LTG12-13,0.088550,False,STG,L,D0024,Decision,LexicalNoDelay,stimulus,sound,ctx_lh_G_temp_sup-Lateral,-65.771716,-8.370915,-2.009006
4,-1.0,D0024_LTG15-26,0.011030,False,Intersection,L,D0024,Decision,LexicalNoDelay,stimulus,sound,Intersection,-58.577098,-13.150433,-14.837984


In [12]:
# load a exlude pandas df
fpath = '../results/exlude_insula.csv'
exclude_df = pd.read_csv(fpath, index_col=0)
# exclude NaN columns and flatten into single list
exclude_chn = [electrode for item in exclude_df.Exclude.dropna() 
                for electrode in eval(item)]

HGAs = HGAs[~HGAs.channel.isin(exclude_chn)]
print(exclude_chn)

['D0040_L1IF2-3', 'D0040_L1IF3-4', 'D0079_LFAI1-2', 'D0079_LFAI2-3', 'D0079_LFMI1-2', 'D0079_LPI9-10', 'D0103_LAI7-8', 'D0103_LAI3-4', 'D0103_LAI5-6', 'D0103_LAI6-7']


In [13]:
# Insula region classification function for row-wise application
def classify_insula_row(row, y_threshold=0):
    """
    Classify a single insula electrode into AIC or PIC
    Designed for use with df.apply()
    
    Returns: 'AIC', 'PIC', or original roi if not insula
    """
    if row['roi'] != 'Insula':
        return row['roi']
    
    label = row['label']
    y_coord = row['y']
    
    # AIC classification conditions
    if ('G_insular_short' in label or 
        'S_circular_insula_ant' in label or
        ('S_circular_insula_sup' in label and y_coord > y_threshold) or
        ('S_circular_insula_inf' in label and y_coord > y_threshold)):
        return 'AIC'
    
    # PIC classification conditions  
    elif ('G_Ins_lg_and_S_cent_ins' in label or
          ('S_circular_insula_sup' in label and y_coord <= y_threshold) or
          ('S_circular_insula_inf' in label and y_coord <= y_threshold)):
        return 'PIC'
    
    # If no conditions match, return original
    return row['roi']

# Apply classification to entire DataFrame (function handles filtering internally)
HGAs['roi'] = HGAs.apply(classify_insula_row, axis=1, y_threshold=0)

# Statistics results
print(f"AIC electrode count: {len(HGAs[HGAs['roi'] == 'AIC'])}")
print(f"PIC electrode count: {len(HGAs[HGAs['roi'] == 'PIC'])}")

# Check unclassified electrodes
unclassified = HGAs[HGAs['roi'] == 'Insula']
if len(unclassified) > 0:
    print(f"Unclassified electrode count: {len(unclassified)}")
    print("Unclassified label types:", unclassified['label'].unique())
    
HGAs.head()

AIC electrode count: 207104
PIC electrode count: 180416


,time,channel,value,mask,roi,hemi,subject,description,task,phase,modality,label,x,y,z
0,-1.0,D0024_LTG14-15,0.007910,False,STG,L,D0024,Decision,LexicalNoDelay,stimulus,sound,ctx_lh_G_temp_sup-Lateral,-66.294399,-28.103954,7.090004
1,-1.0,D0024_LTG8-9,-0.001315,False,Intersection,L,D0024,Decision,LexicalNoDelay,stimulus,sound,Intersection,-55.427594,-7.961854,2.275063
2,-1.0,D0024_LTG5-6,0.026363,False,SMC,L,D0024,Decision,LexicalNoDelay,stimulus,sound,ctx_lh_G_and_S_subcentral,-65.187402,-14.882005,13.114168
3,-1.0,D0024_LTG12-13,0.088550,False,STG,L,D0024,Decision,LexicalNoDelay,stimulus,sound,ctx_lh_G_temp_sup-Lateral,-65.771716,-8.370915,-2.009006
4,-1.0,D0024_LTG15-26,0.011030,False,Intersection,L,D0024,Decision,LexicalNoDelay,stimulus,sound,Intersection,-58.577098,-13.150433,-14.837984


# Load data

In [17]:
bids_root = '/cwork/ns458/BIDS-1.0_LexicalDecRepDelay/BIDS/'

In [44]:
# load label array
phases = ['Delay', 'Go', 'Response']
conditions = ['Decision','Repeat']

epos = []

subset = HGAs[
    (HGAs.roi.isin(['AIC']))
    &(HGAs.task=='LexicalDelay')
]

for phase in tqdm(phases):
    
    for sub in subset.subject.unique():
        
        picks = subset[subset.subject==sub].channel.unique().tolist()
        
        pt = BIDSPath(
            root=os.path.join(bids_root, 'derivatives', 'epoch(bipolar)'),
            subject=sub,
            datatype='epoch(band)(zscore)',
            processing=phase,
            suffix='highgamma',
            extension='.h5',
            check=False
        )

        for condition in conditions:
            epo_path = pt.update(description=condition).match()[0]
            dc_epo = mne.read_epochs(epo_path, verbose='error')
            onset = dc_epo.events[:, 0] / 2048
            dc_epo.metadata = pd.DataFrame({'onset': onset}, index=dc_epo.selection)
            dc_epo.pick(picks)
            df = dc_epo.to_data_frame(long_format=True, scalings={'seeg': 1}, verbose=False)
            meta = dc_epo.metadata.reset_index(names='epoch')  # epoch == selection
            df = df.merge(meta, on='epoch', how='left')

            df['phase'] = phase
            df['description'] = condition
            df['subject'] = sub
            # df['meta'] = df['condition'].str.rsplit('/', n=1).str[-1]
            # df['condition'] = df['condition'].str.rsplit('/', n=1).str[0]
            epos.append(df)

epos = pd.concat(epos, ignore_index=True)
epos.head()

  0%|          | 0/3 [00:00<?, ?it/s]

Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding m

 33%|███▎      | 1/3 [00:06<00:13,  6.99s/it]

Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding m

 67%|██████▋   | 2/3 [00:13<00:06,  6.99s/it]

Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding metadata with 1 columns
Adding m

100%|██████████| 3/3 [00:21<00:00,  7.01s/it]


,condition,epoch,time,channel,ch_type,value,onset,phase,description,subject
0,Delay/Yes_No/Word/minus/CORRECT,7,-1.000000,D0027_LAI2-3,seeg,-1.155979,877.211914,Delay,Decision,D0027
1,Delay/Yes_No/Word/minus/CORRECT,7,-1.000000,D0027_LAI3-4,seeg,-0.290565,877.211914,Delay,Decision,D0027
2,Delay/Yes_No/Word/minus/CORRECT,7,-0.992188,D0027_LAI2-3,seeg,-1.079248,877.211914,Delay,Decision,D0027
3,Delay/Yes_No/Word/minus/CORRECT,7,-0.992188,D0027_LAI3-4,seeg,-0.414689,877.211914,Delay,Decision,D0027
4,Delay/Yes_No/Word/minus/CORRECT,7,-0.984375,D0027_LAI2-3,seeg,-1.029100,877.211914,Delay,Decision,D0027


In [45]:
epoch_tbl = (
    epos[['subject','condition', 'description', 'phase', 'epoch', 'onset']]
    .drop_duplicates()
    .sort_values(['subject', 'epoch', 'onset'])
)

In [70]:
# 用 merge_asof 按 onset 时间匹配：每个 Go/Resp 找到它之前最近的 Delay
epoch_tbl = (
    epos[['subject', 'description', 'phase', 'epoch', 'onset']]
    .drop_duplicates()
)

delay = epoch_tbl[epoch_tbl['phase'].eq('Delay')][['subject','description','epoch','onset']].rename(columns={'epoch':'delay_epoch','onset':'delay_onset'})
go    = epoch_tbl[epoch_tbl['phase'].eq('Go')][['subject','description','onset']].rename(columns={'onset':'go_onset'})
resp  = epoch_tbl[epoch_tbl['phase'].eq('Response')][['subject','description','onset']].rename(columns={'onset':'resp_onset'})

# 给每个 Delay 分配唯一 trial_idx（按 subject + description 分组）
delay = delay.sort_values(['delay_onset','subject','description'], kind='mergesort').reset_index(drop=True)
delay['trial_idx'] = delay.groupby(['subject','description']).cumcount()

# merge_asof 需要 left/right 在 on 上全局严格排序
go = go.dropna(subset=['go_onset']).sort_values(['go_onset','subject','description'], kind='mergesort').reset_index(drop=True)
resp = resp.dropna(subset=['resp_onset']).sort_values(['resp_onset','subject','description'], kind='mergesort').reset_index(drop=True)

# merge_asof: 每个 Go 找到它之前最近的 Delay
go_matched = pd.merge_asof(
    go,
    delay[['subject','description','delay_onset','trial_idx']],
    by=['subject','description'],
    left_on='go_onset',
    right_on='delay_onset',
    direction='backward'
)

resp_matched = pd.merge_asof(
    resp,
    delay[['subject','description','delay_onset','trial_idx']],
    by=['subject','description'],
    left_on='resp_onset',
    right_on='delay_onset',
    direction='backward'
)

# 合并到 delay 表
trials = (
    delay
    .merge(go_matched[['subject','description','trial_idx','go_onset']], on=['subject','description','trial_idx'], how='left')
    .merge(resp_matched[['subject','description','trial_idx','resp_onset']], on=['subject','description','trial_idx'], how='left')
)

trials['rt'] = trials['resp_onset'] - trials['go_onset']
trials.head(10)

,subject,description,delay_epoch,delay_onset,trial_idx,go_onset,resp_onset,rt
0,D0079,Decision,2,12.631348,0,13.876465,14.775879,0.899414
1,D0079,Decision,7,19.450684,1,20.774414,21.610840,0.836426
2,D0079,Decision,12,26.445312,2,27.509277,28.125488,0.616211
3,D0079,Repeat,17,33.200195,0,34.412109,35.350098,0.937988
4,D0079,Repeat,22,40.022949,1,41.127441,42.136719,1.009277
5,D0079,Repeat,27,46.877930,2,48.028320,48.828125,0.799805
6,D0079,Decision,32,53.933594,3,55.145020,56.033691,0.888672
7,D0079,Repeat,37,60.837402,3,62.028809,63.047363,1.018555
8,D0079,Repeat,42,67.639648,4,68.793457,69.917969,1.124512
9,D0079,Repeat,47,74.533203,5,75.776855,76.883301,1.106445


In [80]:
# 用 merge_asof 按 onset 时间匹配：每个 Go/Resp 找到它之前最近的 Delay
epoch_tbl = (
    epos[['subject', 'description', 'phase', 'epoch', 'onset', 'condition']]
    .drop_duplicates()
)

delay = epoch_tbl[epoch_tbl['phase'].eq('Delay')][['subject','description','condition','epoch','onset']].rename(columns={'epoch':'delay_epoch','onset':'delay_onset'})
go    = epoch_tbl[epoch_tbl['phase'].eq('Go')][['subject','description','epoch','onset']].rename(columns={'epoch':'go_epoch','onset':'go_onset'})
resp  = epoch_tbl[epoch_tbl['phase'].eq('Response')][['subject','description','epoch','onset']].rename(columns={'epoch':'resp_epoch','onset':'resp_onset'})

# 给每个 Delay 分配唯一 trial_idx（按 subject + description 分组）
delay = delay.sort_values(['delay_onset','subject','description'], kind='mergesort').reset_index(drop=True)
delay['trial_idx'] = delay.groupby(['subject','description']).cumcount()

# merge_asof 需要 left/right 在 on 上全局严格排序
go = go.dropna(subset=['go_onset']).sort_values(['go_onset','subject','description'], kind='mergesort').reset_index(drop=True)
resp = resp.dropna(subset=['resp_onset']).sort_values(['resp_onset','subject','description'], kind='mergesort').reset_index(drop=True)

# merge_asof: 每个 Go 找到它之前最近的 Delay
go_matched = pd.merge_asof(
    go,
    delay[['subject','description','delay_onset','trial_idx']],
    by=['subject','description'],
    left_on='go_onset',
    right_on='delay_onset',
    direction='backward'
)

resp_matched = pd.merge_asof(
    resp,
    delay[['subject','description','delay_onset','trial_idx']],
    by=['subject','description'],
    left_on='resp_onset',
    right_on='delay_onset',
    direction='backward'
)

# 合并到 delay 表（保留 condition）
trials = (
    delay
    .merge(go_matched[['subject','description','trial_idx','go_onset','go_epoch']], on=['subject','description','trial_idx'], how='left')
    .merge(resp_matched[['subject','description','trial_idx','resp_onset','resp_epoch']], on=['subject','description','trial_idx'], how='left')
)

trials['rt'] = trials['resp_onset'] - trials['go_onset']
trials.head(10)

,subject,description,condition,delay_epoch,delay_onset,trial_idx,go_onset,go_epoch,resp_onset,resp_epoch,rt
0,D0079,Decision,Delay/Yes_No/Word/petal/CORRECT,2,12.631348,0,13.876465,3.0,14.775879,4.0,0.899414
1,D0079,Decision,Delay/Yes_No/Word/cabin/CORRECT,7,19.450684,1,20.774414,8.0,21.610840,9.0,0.836426
2,D0079,Decision,Delay/Yes_No/Word/tenet/CORRECT,12,26.445312,2,27.509277,13.0,28.125488,14.0,0.616211
3,D0079,Repeat,Delay/Repeat/Word/petal/CORRECT,17,33.200195,0,34.412109,18.0,35.350098,19.0,0.937988
4,D0079,Repeat,Delay/Repeat/Nonword/delin/CORRECT,22,40.022949,1,41.127441,23.0,42.136719,24.0,1.009277
5,D0079,Repeat,Delay/Repeat/Word/rival/CORRECT,27,46.877930,2,48.028320,28.0,48.828125,29.0,0.799805
6,D0079,Decision,Delay/Yes_No/Nonword/lomic/CORRECT,32,53.933594,3,55.145020,33.0,56.033691,34.0,0.888672
7,D0079,Repeat,Delay/Repeat/Word/manic/CORRECT,37,60.837402,3,62.028809,38.0,63.047363,39.0,1.018555
8,D0079,Repeat,Delay/Repeat/Nonword/herib/CORRECT,42,67.639648,4,68.793457,43.0,69.917969,44.0,1.124512
9,D0079,Repeat,Delay/Repeat/Nonword/labin/CORRECT,47,74.533203,5,75.776855,48.0,76.883301,49.0,1.106445


In [ ]:
# 把 RT merge 回包含 Delay/Go/Response 的长表
# 构建 phase+epoch -> idx/rt/condition 的映射
phase_map = pd.concat([
    trials[['subject','description','trial_idx','condition','rt','delay_epoch']]
        .rename(columns={'trial_idx':'idx','delay_epoch':'epoch'})
        .assign(phase='Delay'),
    trials[['subject','description','trial_idx','condition','rt','go_epoch']]
        .rename(columns={'trial_idx':'idx','go_epoch':'epoch'})
        .assign(phase='Go'),
    trials[['subject','description','trial_idx','condition','rt','resp_epoch']]
        .rename(columns={'trial_idx':'idx','resp_epoch':'epoch'})
        .assign(phase='Response'),
], ignore_index=True)

# 合并回 epos（保留所有 phase）
epos = (
    epos
    .merge(
        phase_map,
        on=['subject','description','phase','epoch'],
        how='left',
        suffixes=('', '_trial')
    )
    .drop(columns=['condition_trial'])
)

epos.head()

,condition,epoch,time,channel,ch_type,value,onset,phase,description,subject,idx,rt
0,Delay/Yes_No/Word/minus/CORRECT,7,-1.000000,D0027_LAI2-3,seeg,-1.155979,877.211914,Delay,Decision,D0027,0,0.108887
1,Delay/Yes_No/Word/minus/CORRECT,7,-1.000000,D0027_LAI3-4,seeg,-0.290565,877.211914,Delay,Decision,D0027,0,0.108887
2,Delay/Yes_No/Word/minus/CORRECT,7,-0.992188,D0027_LAI2-3,seeg,-1.079248,877.211914,Delay,Decision,D0027,0,0.108887
3,Delay/Yes_No/Word/minus/CORRECT,7,-0.992188,D0027_LAI3-4,seeg,-0.414689,877.211914,Delay,Decision,D0027,0,0.108887
4,Delay/Yes_No/Word/minus/CORRECT,7,-0.984375,D0027_LAI2-3,seeg,-1.029100,877.211914,Delay,Decision,D0027,0,0.108887


In [73]:
trials[trials.subject == 'D0028']

,subject,description,delay_epoch,delay_onset,trial_idx,go_onset,resp_onset,rt
1049,D0028,Repeat,2,1188.794922,0,1189.944824,1191.014648,1.069824
1064,D0028,Decision,7,1195.007324,0,1196.083984,1195.916992,-0.166992
1079,D0028,Decision,12,1201.457031,1,1202.594238,1203.097168,0.502930
1094,D0028,Repeat,17,1207.917969,1,1209.125977,1209.857910,0.731934
1109,D0028,Repeat,22,1214.261230,2,1215.495605,1216.082520,0.586914
...,...,...,...,...,...,...,...,...
5240,D0028,Repeat,1587,3492.321289,155,3493.397949,3494.416016,1.018066
5245,D0028,Repeat,1592,3498.829590,156,3499.969238,3501.039551,1.070312
5251,D0028,Repeat,1597,3505.495117,157,3506.538086,3508.015625,1.477539
5257,D0028,Decision,1602,3512.047852,161,3513.167969,3514.458008,1.290039


In [62]:
epoch_tbl = (
    epos[['subject', 'description', 'phase', 'condition', 'epoch', 'onset']]
    .drop_duplicates()
    .sort_values(['subject','epoch', 'onset', 'condition','description'])
)


In [64]:
epoch_tbl

,subject,description,phase,condition,epoch,onset
107520,D0027,Repeat,Delay,Delay/Repeat/Nonword/galel/EARLY_RESP,2,870.421875
11365760,D0027,Repeat,Response,Resp/Repeat/Nonword/galel/EARLY_RESP,3,871.597656
5736640,D0027,Repeat,Go,Go/Repeat/Nonword/galel/EARLY_RESP,4,871.695801
0,D0027,Decision,Delay,Delay/Yes_No/Word/minus/CORRECT,7,877.211914
5629120,D0027,Decision,Go,Go/Yes_No/Word/minus/CORRECT,8,878.362793
...,...,...,...,...,...,...
11203840,D0103,Decision,Go,Go/Yes_No/Word/libel/CORRECT,1673,3473.769531
16831360,D0103,Decision,Response,Resp/Yes_No/Word/libel/CORRECT,1674,3475.562012
5575040,D0103,Decision,Delay,Delay/Yes_No/Nonword/berin/CORRECT,1677,3479.258301
11204160,D0103,Decision,Go,Go/Yes_No/Nonword/berin/CORRECT,1678,3480.502930
